In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORKSPACE = Path('/kaggle/working')
REPO_NAME = 'point_cloud_playground_play'
PROJECT_DIR = WORKSPACE / REPO_NAME
PRIVATE_REPO = 'github.com/kslannmnd/point_cloud_playground_play.git'
GITHUB_TOKEN_SECRET = 'GITHUB_TOKEN'

S3DIS_PREPROCESSED_DIR = Path('/kaggle/input/datasets/ayukiseleva/s3dis-preprocessed')
R3D_FILE = None  # Example: Path('/kaggle/input/my-r3d-dataset/2026-03-24--23-18-48.r3d')
R3D_BBOX_CSV = None  # Optional CSV with x_min,y_min,z_min,x_max,y_max,z_max,label,color

RUN_ENV_SETUP = True
RUN_TRAINING = True
RUN_ROOM_INFERENCE_AND_METRICS = True
RUN_ALL_ROOM_INFERENCE = False
RUN_R3D_INFERENCE = R3D_FILE is not None

AREA = 'Area_1'
ROOM = 'office_1'
OBJECT_NAME = 'chair_1'

def run(cmd, cwd=None, env=None, check=True):
    print('\n[RUN]', ' '.join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else cmd, '\n')
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, shell=isinstance(cmd, str), check=check)

assert S3DIS_PREPROCESSED_DIR.exists(), f'Missing preprocessed S3DIS dataset: {S3DIS_PREPROCESSED_DIR}'

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
except Exception as exc:
    raise RuntimeError(f'Create Kaggle secret {GITHUB_TOKEN_SECRET!r} with private repo access first.') from exc

clone_url = f'https://{token}@{PRIVATE_REPO}'
if PROJECT_DIR.exists():
    print('Project already exists:', PROJECT_DIR)
else:
    run(['git', 'clone', clone_url, str(PROJECT_DIR)])

run(['git', 'remote', 'set-url', 'origin', f'https://{PRIVATE_REPO}'], cwd=PROJECT_DIR, check=False)
print('Project dir:', PROJECT_DIR)

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'wheel', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_DIR)

COMMON = [
    'env=kaggle',
    'dataset=s3dis_preprocessed',
    'inference=s3dis',
    'dataset.preprocessed.source_dir=/kaggle/input/datasets/ayukiseleva/s3dis-preprocessed',
]

In [ ]:
if RUN_ENV_SETUP:
    run([sys.executable, 'scripts/prepare_kaggle_env.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/install_dependencies_and_build_softgroup.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/prepare_s3dis.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/download_softgroup_checkpoints.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/patch_softgroup_for_kaggle.py', *COMMON], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/generate_softgroup_configs.py', *COMMON], cwd=PROJECT_DIR)
else:
    print('RUN_ENV_SETUP=False')

In [ ]:
if RUN_TRAINING:
    # Parameter-only experiment.
    run([
        sys.executable, 'scripts/train.py', *COMMON,
        'training=template_train',
        'training.experiment_name=kaggle_template_train',
        'training.epochs=2',
        'training.lr=0.001',
        'training.train_batch_size=1',
        'training.num_workers=2',
    ], cwd=PROJECT_DIR)
else:
    print('RUN_TRAINING=False')

In [ ]:
# Architecture-level experiment hook. Edit experiments/train_exp_1.py in the repo,
# then run this cell to execute the external experiment file with Hydra params.
RUN_EXTERNAL_ARCH_EXPERIMENT = False
if RUN_EXTERNAL_ARCH_EXPERIMENT:
    run([
        sys.executable, 'scripts/train.py', *COMMON,
        'training=external_exp',
        'training.experiment_name=kaggle_arch_exp_1',
        'training.epochs.backbone=2',
        'training.epochs.full_softgroup=2',
    ], cwd=PROJECT_DIR)

In [ ]:
CHECKPOINT = 'outputs/checkpoints/kaggle_template_train_full_softgroup_latest.pth'
if not (PROJECT_DIR / CHECKPOINT).exists():
    CHECKPOINT = 'outputs/checkpoints/softgroup_s3dis_pretrained.pth'
print('Using checkpoint:', CHECKPOINT)

In [ ]:
if RUN_ROOM_INFERENCE_AND_METRICS:
    run([
        sys.executable, 'scripts/infer.py', *COMMON,
        f'inference.checkpoint={CHECKPOINT}',
        'inference.target.kind=room',
        f'inference.target.area={AREA}',
        f'inference.target.room={ROOM}',
    ], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/show_gt_compare_table.py', AREA, ROOM, *COMMON, f'inference.checkpoint={CHECKPOINT}'], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/show_room_bbox.py', AREA, ROOM, *COMMON, f'inference.checkpoint={CHECKPOINT}'], cwd=PROJECT_DIR)
    run([sys.executable, 'scripts/show_gt_compare_plot.py', AREA, ROOM, OBJECT_NAME, *COMMON, f'inference.checkpoint={CHECKPOINT}'], cwd=PROJECT_DIR, check=False)
else:
    print('RUN_ROOM_INFERENCE_AND_METRICS=False')

In [ ]:
if RUN_ALL_ROOM_INFERENCE:
    run([
        sys.executable, 'scripts/infer.py', *COMMON,
        f'inference.checkpoint={CHECKPOINT}',
        'inference.target.kind=all',
    ], cwd=PROJECT_DIR)
else:
    print('RUN_ALL_ROOM_INFERENCE=False')

In [ ]:
from IPython.display import HTML, display
html_root = PROJECT_DIR / 'outputs' / 'visualizations' / 's3dis'
if html_root.exists():
    html_files = sorted(html_root.glob('*.html'))
    print('Saved HTML visualizations:', len(html_files))
    if html_files:
        print(html_files[-1])
        display(HTML(html_files[-1].read_text(encoding='utf-8')))
else:
    print('No visualization directory yet:', html_root)

In [ ]:
if RUN_R3D_INFERENCE:
    cmd = [
        sys.executable, 'run_r3d_inference.py', str(R3D_FILE),
        'dataset=r3d',
        'inference=r3d',
        f'inference.checkpoint={CHECKPOINT}',
    ]
    if R3D_BBOX_CSV is not None:
        cmd.append(f'dataset.bbox.csv_path={R3D_BBOX_CSV}')
    run(cmd, cwd=PROJECT_DIR)
else:
    print('RUN_R3D_INFERENCE=False. Set R3D_FILE to a Kaggle input .r3d path to run this cell.')

In [ ]:
r3d_root = PROJECT_DIR / 'outputs' / 'r3d_inference'
if r3d_root.exists():
    runs = sorted([p for p in r3d_root.iterdir() if p.is_dir()])
    if runs:
        html_path = runs[-1] / 'point_cloud.html'
        print('Latest R3D visualization:', html_path)
        if html_path.exists():
            display(HTML(html_path.read_text(encoding='utf-8')))
else:
    print('No R3D inference outputs yet.')